In [ ]:
import geopandas as gpd
import pandas as pd

from openplaces.api import get_admin, read_entities
from openplaces.geo.vector import points_from_coords
from openplaces.io import share
from openplaces.path import external_path, share_path

In [ ]:
BUILDINGS_CHEER_PATH = external_path(
    'US-NC', 'building-cheer-v0', filename='Inventory_v0_NC.parquet'
)

CHEER_ADMIN3_IDS = (
    'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN '
    'US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL '
    'US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA '
    'US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB '
    'US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'
).split(' ')

PARCEL_COLUMNS_TO_LINK = [
    'parcel_id_admin3',
    'year_built',
    'improvement_value',
    'land_value',
]

# Join buildings to unique parcels

Runs in 44 sec

In [ ]:
building_inventory_list = []
for admin3_id in CHEER_ADMIN3_IDS:
    print(admin3_id, end=', ')

    # Read CHEER inventory
    county_fips = get_admin(admin3_id)['admin3_id_admin1'].values[0]
    buildings_cheer = gpd.read_parquet(
        BUILDINGS_CHEER_PATH, filters=[('county', '==', county_fips)]
    ).set_index('bid')

    # Read NCOneMap parcels
    parcels = read_entities('US-NC_parcel-nconemap-2025', admin3_id, geom=True)
    parcels = parcels[~parcels.drop(columns=['source_deed']).duplicated()].copy()

    # Duplicate parcels are often actual parcels for which the polygon
    # hasn't been carved out of a larger polygon yet - these need to
    # be linked with addresses or through other means. Flag for later.
    parcels['has_duplicate'] = parcels['geo_id'].duplicated(keep=False)
    unique_parcels = parcels[~parcels['geo_id'].duplicated()].copy()

    # After dropping geo_id duplicates, use 'geo_id' as 'parcel_id'
    unique_parcels.index = unique_parcels['geo_id'].rename('parcel_id')

    # Spatially join building centroids with parcels
    buildings_cheer_on_parcels = gpd.sjoin(
        points_from_coords(buildings_cheer),
        unique_parcels[PARCEL_COLUMNS_TO_LINK + ['geometry']],
        how='left',
    )

    # Flag buildings without a parcel
    mask_no_parcel = buildings_cheer_on_parcels['parcel_id'].isnull()
    buildings_cheer_on_parcels.loc[mask_no_parcel, 'issue'] = 'no parcel'

    # Flag buildings linked to duplicated parcels and remove values
    # (for now)
    mask_duplicates = buildings_cheer_on_parcels['parcel_id'].isin(
        parcels[parcels['has_duplicate']]['geo_id'].unique()
    )
    buildings_cheer_on_parcels.loc[mask_duplicates, PARCEL_COLUMNS_TO_LINK] = None
    buildings_cheer_on_parcels.loc[mask_duplicates, 'issue'] = 'duplicate parcel'

    # Flag buildings linked to parcels with multiple buildings
    mask_multiple_buildings = (
        ~mask_duplicates
        & buildings_cheer_on_parcels['parcel_id'].notnull()
        & buildings_cheer_on_parcels['parcel_id'].duplicated(keep=False)
    )
    buildings_cheer_on_parcels.loc[mask_multiple_buildings, 'issue'] = (
        'multi-building value'
    )

    building_inventory_list += [buildings_cheer_on_parcels]

building_inventory = pd.concat(building_inventory_list)

# Share inventory

In [ ]:
out_path = share_path('US-NC', 'building-cheer-v01', filename='Eastern')

share(building_inventory, out_path, 'share/2026/cheer')